In [ ]:
import json
from datetime import datetime

def export_validation_results(results, output_path=None):
    """
    Export validation results to JSON file
    
    Args:
        results: Validation results dictionary
        output_path: Path to save the JSON file (optional)
    """
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"validation_report_{timestamp}.json"
    
    # Convert defaultdict to regular dict for JSON serialization
    export_data = {
        'timestamp': datetime.now().isoformat(),
        'stats': dict(results['stats']),
        'errors': {k: list(v) for k, v in results['errors'].items()},
        'warnings': {k: list(v) for k, v in results['warnings'].items()}
    }
    
    with open(output_path, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"Validation results exported to: {output_path}")
    return output_path

# Uncomment to export results
# export_validation_results(archive_results, "archive_validation_report.json")
# export_validation_results(dataset_results, "dataset_validation_report.json")

## Export Validation Results

Save validation results to a file for later review:

In [ ]:
def analyze_class_distribution_detailed(dataset_path, splits=['train', 'test', 'valid']):
    """
    Detailed class distribution analysis
    
    Args:
        dataset_path: Root path to the dataset
        splits: List of dataset splits to analyze
    """
    # Load class names
    yaml_path = os.path.join(dataset_path, 'data.yaml')
    try:
        with open(yaml_path, 'r') as f:
            data_config = yaml.safe_load(f)
            class_names = data_config.get('names', [])
    except:
        class_names = []
    
    print(f"{'='*70}")
    print(f"CLASS DISTRIBUTION - {os.path.basename(dataset_path)}")
    print(f"{'='*70}\n")
    
    for split in splits:
        labels_dir = os.path.join(dataset_path, split, 'labels')
        if not os.path.exists(labels_dir):
            print(f"{split.upper()}: Directory not found\n")
            continue
        
        class_counts = defaultdict(int)
        image_counts = defaultdict(int)
        
        for label_file in Path(labels_dir).glob("*.txt"):
            try:
                with open(label_file, 'r') as f:
                    seen_classes = set()
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            class_id = int(parts[0])
                            class_counts[class_id] += 1
                            seen_classes.add(class_id)
                    
                    # Count images per class
                    for cls in seen_classes:
                        image_counts[cls] += 1
            except:
                continue
        
        total_annotations = sum(class_counts.values())
        total_images = len(list(Path(labels_dir).glob("*.txt")))
        
        print(f"{split.upper()}:")
        print(f"  Total images: {total_images}")
        print(f"  Total annotations: {total_annotations}")
        
        if class_counts:
            print(f"\n  Class breakdown:")
            print(f"  {'ID':<4} {'Class Name':<30} {'Annotations':<12} {'Images':<8} {'Ann%':<8} {'Img%'}")
            print(f"  {'-'*80}")
            
            for class_id in sorted(class_counts.keys()):
                count = class_counts[class_id]
                img_count = image_counts[class_id]
                ann_pct = (count / total_annotations * 100) if total_annotations > 0 else 0
                img_pct = (img_count / total_images * 100) if total_images > 0 else 0
                
                class_name = class_names[class_id] if class_id < len(class_names) else f"Unknown-{class_id}"
                
                print(f"  {class_id:<4} {class_name:<30} {count:<12} {img_count:<8} {ann_pct:>6.1f}% {img_pct:>6.1f}%")
        
        print()

# Analyze both datasets
print("Archive Dataset (Rock Paper Scissors):")
analyze_class_distribution_detailed(archive_path, splits=['train', 'test', 'valid'])

print("\n" + "="*70 + "\n")

print("Main Dataset:")
analyze_class_distribution_detailed(dataset_path, splits=['train', 'test'])

## Class Distribution Visualization

Analyze the distribution of classes across datasets:

In [ ]:
# Validate the main dataset
dataset_path = r"c:\Users\jhon\Desktop\Coding\AI\AI231-machine-problems\data\mp7\dataset"
dataset_results = validate_dataset(dataset_path, splits=['train', 'test'])
print_validation_report(dataset_results)

## Validate Main Dataset (Products + Rock Paper Scissors)

In [ ]:
# Validate the archive (Rock Paper Scissors) dataset
archive_path = r"c:\Users\jhon\Desktop\Coding\AI\AI231-machine-problems\data\mp7\archive (1)"
archive_results = validate_dataset(archive_path, splits=['train', 'test', 'valid'])
print_validation_report(archive_results)

## Validate Archive Dataset (Rock Paper Scissors)

In [4]:
import os
import yaml
from pathlib import Path
from collections import defaultdict
from PIL import Image
import traceback

def validate_dataset(dataset_path, splits=['train', 'test', 'valid']):
    """
    Validate images and labels in a YOLO dataset
    
    Args:
        dataset_path: Root path to the dataset
        splits: List of dataset splits to validate
    
    Returns:
        Dictionary containing validation results
    """
    results = {
        'summary': {},
        'errors': defaultdict(list),
        'warnings': defaultdict(list),
        'stats': {}
    }
    
    # Load data.yaml to get class information
    yaml_path = os.path.join(dataset_path, 'data.yaml')
    try:
        with open(yaml_path, 'r') as f:
            data_config = yaml.safe_load(f)
            num_classes = data_config.get('nc', 0)
            class_names = data_config.get('names', [])
            print(f"Dataset: {os.path.basename(dataset_path)}")
            print(f"Classes: {num_classes}")
            print(f"Splits to validate: {splits}\n")
    except Exception as e:
        results['errors']['config'].append(f"Failed to load data.yaml: {e}")
        return results
    
    for split in splits:
        print(f"{'='*70}")
        print(f"Validating {split.upper()} split...")
        print(f"{'='*70}")
        
        images_dir = os.path.join(dataset_path, split, 'images')
        labels_dir = os.path.join(dataset_path, split, 'labels')
        
        # Check if directories exist
        if not os.path.exists(images_dir):
            results['errors'][split].append(f"Images directory not found: {images_dir}")
            print(f"❌ Images directory not found\n")
            continue
        
        if not os.path.exists(labels_dir):
            results['errors'][split].append(f"Labels directory not found: {labels_dir}")
            print(f"❌ Labels directory not found\n")
            continue
        
        # Get all image and label files
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
        image_files = {}
        for ext in image_extensions:
            for img_path in Path(images_dir).glob(f"*{ext}"):
                image_files[img_path.stem] = img_path
            for img_path in Path(images_dir).glob(f"*{ext.upper()}"):
                image_files[img_path.stem] = img_path
        
        label_files = {lbl.stem: lbl for lbl in Path(labels_dir).glob("*.txt")}
        
        print(f"Found {len(image_files)} images")
        print(f"Found {len(label_files)} labels")
        
        # Initialize counters
        stats = {
            'total_images': len(image_files),
            'total_labels': len(label_files),
            'valid_pairs': 0,
            'missing_labels': 0,
            'missing_images': 0,
            'corrupted_images': 0,
            'invalid_labels': 0,
            'empty_labels': 0,
            'bbox_errors': 0,
            'class_errors': 0,
            'total_annotations': 0
        }
        
        # Check for missing labels
        missing_labels = set(image_files.keys()) - set(label_files.keys())
        if missing_labels:
            stats['missing_labels'] = len(missing_labels)
            for name in list(missing_labels)[:5]:  # Show first 5
                results['warnings'][split].append(f"Missing label for image: {name}")
            if len(missing_labels) > 5:
                results['warnings'][split].append(f"... and {len(missing_labels) - 5} more missing labels")
        
        # Check for missing images
        missing_images = set(label_files.keys()) - set(image_files.keys())
        if missing_images:
            stats['missing_images'] = len(missing_images)
            for name in list(missing_images)[:5]:  # Show first 5
                results['warnings'][split].append(f"Missing image for label: {name}")
            if len(missing_images) > 5:
                results['warnings'][split].append(f"... and {len(missing_images) - 5} more missing images")
        
        # Validate image-label pairs
        common_names = set(image_files.keys()) & set(label_files.keys())
        
        for name in common_names:
            img_path = image_files[name]
            lbl_path = label_files[name]
            
            # Validate image
            try:
                with Image.open(img_path) as img:
                    img_width, img_height = img.size
                    img.verify()  # Verify image integrity
            except Exception as e:
                stats['corrupted_images'] += 1
                results['errors'][split].append(f"Corrupted image {img_path.name}: {str(e)}")
                continue
            
            # Validate label
            try:
                with open(lbl_path, 'r') as f:
                    lines = f.readlines()
                
                if not lines or all(not line.strip() for line in lines):
                    stats['empty_labels'] += 1
                    results['warnings'][split].append(f"Empty label file: {lbl_path.name}")
                    continue
                
                annotation_count = 0
                for line_num, line in enumerate(lines, 1):
                    line = line.strip()
                    if not line:
                        continue
                    
                    parts = line.split()
                    if len(parts) != 5:
                        stats['invalid_labels'] += 1
                        results['errors'][split].append(
                            f"{lbl_path.name}:{line_num} - Invalid format (expected 5 values, got {len(parts)})"
                        )
                        continue
                    
                    try:
                        class_id = int(parts[0])
                        x_center, y_center, width, height = map(float, parts[1:])
                        
                        # Validate class ID
                        if class_id < 0 or class_id >= num_classes:
                            stats['class_errors'] += 1
                            results['errors'][split].append(
                                f"{lbl_path.name}:{line_num} - Invalid class {class_id} (valid range: 0-{num_classes-1})"
                            )
                        
                        # Validate bounding box coordinates (should be normalized 0-1)
                        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 
                                0 <= width <= 1 and 0 <= height <= 1):
                            stats['bbox_errors'] += 1
                            results['errors'][split].append(
                                f"{lbl_path.name}:{line_num} - Invalid bbox coordinates (must be 0-1): "
                                f"x={x_center}, y={y_center}, w={width}, h={height}"
                            )
                        
                        # Check if bbox is within image bounds
                        x_min = x_center - width / 2
                        x_max = x_center + width / 2
                        y_min = y_center - height / 2
                        y_max = y_center + height / 2
                        
                        if x_min < 0 or x_max > 1 or y_min < 0 or y_max > 1:
                            stats['bbox_errors'] += 1
                            results['warnings'][split].append(
                                f"{lbl_path.name}:{line_num} - Bbox extends beyond image bounds"
                            )
                        
                        annotation_count += 1
                        
                    except ValueError as e:
                        stats['invalid_labels'] += 1
                        results['errors'][split].append(
                            f"{lbl_path.name}:{line_num} - Failed to parse values: {str(e)}"
                        )
                
                stats['total_annotations'] += annotation_count
                stats['valid_pairs'] += 1
                
            except Exception as e:
                stats['invalid_labels'] += 1
                results['errors'][split].append(f"Failed to read label {lbl_path.name}: {str(e)}")
        
        # Store stats
        results['stats'][split] = stats
        
        # Print summary for this split
        print(f"\n✓ Valid image-label pairs: {stats['valid_pairs']}")
        print(f"⚠ Missing labels: {stats['missing_labels']}")
        print(f"⚠ Missing images: {stats['missing_images']}")
        print(f"❌ Corrupted images: {stats['corrupted_images']}")
        print(f"❌ Invalid labels: {stats['invalid_labels']}")
        print(f"⚠ Empty labels: {stats['empty_labels']}")
        print(f"❌ Bounding box errors: {stats['bbox_errors']}")
        print(f"❌ Class ID errors: {stats['class_errors']}")
        print(f"📊 Total annotations: {stats['total_annotations']}")
        print()
    
    return results

def print_validation_report(results):
    """Print a detailed validation report"""
    print(f"\n{'='*70}")
    print("VALIDATION REPORT")
    print(f"{'='*70}\n")
    
    # Print errors
    total_errors = sum(len(errors) for errors in results['errors'].values())
    if total_errors > 0:
        print(f"❌ ERRORS FOUND: {total_errors}")
        print("-" * 70)
        for split, errors in results['errors'].items():
            if errors:
                print(f"\n{split.upper()}:")
                for error in errors[:10]:  # Show first 10
                    print(f"  • {error}")
                if len(errors) > 10:
                    print(f"  ... and {len(errors) - 10} more errors")
        print()
    
    # Print warnings
    total_warnings = sum(len(warnings) for warnings in results['warnings'].values())
    if total_warnings > 0:
        print(f"⚠ WARNINGS FOUND: {total_warnings}")
        print("-" * 70)
        for split, warnings in results['warnings'].items():
            if warnings:
                print(f"\n{split.upper()}:")
                for warning in warnings[:10]:  # Show first 10
                    print(f"  • {warning}")
                if len(warnings) > 10:
                    print(f"  ... and {len(warnings) - 10} more warnings")
        print()
    
    # Print overall stats
    print(f"📊 OVERALL STATISTICS")
    print("-" * 70)
    for split, stats in results['stats'].items():
        print(f"\n{split.upper()}:")
        print(f"  Total images: {stats['total_images']}")
        print(f"  Total labels: {stats['total_labels']}")
        print(f"  Valid pairs: {stats['valid_pairs']}")
        print(f"  Total annotations: {stats['total_annotations']}")
        if stats['valid_pairs'] > 0:
            avg_annotations = stats['total_annotations'] / stats['valid_pairs']
            print(f"  Avg annotations per image: {avg_annotations:.2f}")
    
    print(f"\n{'='*70}")
    
    # Overall health check
    if total_errors == 0 and total_warnings == 0:
        print("✅ DATASET IS HEALTHY - No issues found!")
    elif total_errors == 0:
        print("✅ DATASET IS VALID - Only warnings found (non-critical)")
    else:
        print("❌ DATASET HAS ERRORS - Please fix critical issues")
    print(f"{'='*70}\n")

# Dataset Validation Tool

This section validates the integrity of images and labels in the YOLO dataset, checking for:
- Missing image or label files
- Corrupted or unreadable images
- Invalid label format or values
- Bounding box coordinate validation
- Image-label pair matching

## Validation Script

In [ ]:
import os
import yaml
from pathlib import Path

# Define paths
archive_path = r"c:\Users\jhon\Desktop\Coding\AI\AI231-machine-problems\data\mp7\archive (1)"
dataset_yaml = r"c:\Users\jhon\Desktop\Coding\AI\AI231-machine-problems\data\mp7\dataset\data.yaml"

# Read the dataset data.yaml to get the class mapping
with open(dataset_yaml, 'r') as f:
    dataset_info = yaml.safe_load(f)

# Current class indices in archive (1) dataset
old_class_mapping = {
    0: 'Rock',      # Old class 0
    1: 'Paper',     # Old class 1
    2: 'Scissors'   # Old class 2
}

# New class indices based on dataset data.yaml
new_class_mapping = {
    'Rock': 34,
    'Paper': 35,
    'Scissors': 36  # Adding Scissors as class 36
}

print("Original dataset classes:")
for idx, name in enumerate(dataset_info['names']):
    print(f"  {idx}: {name}")

print(f"\nOld class mapping (archive dataset):")
for old_idx, class_name in old_class_mapping.items():
    print(f"  {old_idx}: {class_name}")

print(f"\nNew class mapping:")
for class_name, new_idx in new_class_mapping.items():
    print(f"  {class_name} -> {new_idx}")

def update_label_files(labels_dir, old_to_new_mapping):
    """
    Update YOLO label files with new class indices
    
    Args:
        labels_dir: Directory containing label files
        old_to_new_mapping: Dictionary mapping old class indices to new ones
    """
    label_files = list(Path(labels_dir).glob("*.txt"))
    updated_count = 0
    
    for label_file in label_files:
        try:
            # Read the label file
            with open(label_file, 'r') as f:
                lines = f.readlines()
            
            # Update class indices
            updated_lines = []
            for line in lines:
                parts = line.strip().split()
                if parts:
                    old_class = int(parts[0])
                    if old_class in old_to_new_mapping:
                        # Replace old class with new class
                        parts[0] = str(old_to_new_mapping[old_class])
                        updated_lines.append(' '.join(parts) + '\n')
                        updated_count += 1
                    else:
                        updated_lines.append(line)
            
            # Write back the updated content
            with open(label_file, 'w') as f:
                f.writelines(updated_lines)
                
        except Exception as e:
            print(f"Error processing {label_file}: {e}")
    
    return updated_count, len(label_files)

# Create mapping from old indices to new indices
old_to_new = {
    0: new_class_mapping['Rock'],
    1: new_class_mapping['Paper'],
    2: new_class_mapping['Scissors']
}

print(f"\n{'='*60}")
print("Starting label update process...")
print(f"{'='*60}\n")

# Update labels in all splits (train, test, valid)
splits = ['train', 'test', 'valid']
total_updated = 0
total_files = 0

for split in splits:
    labels_dir = os.path.join(archive_path, split, 'labels')
    if os.path.exists(labels_dir):
        print(f"Processing {split} split...")
        updated, files = update_label_files(labels_dir, old_to_new)
        total_updated += updated
        total_files += files
        print(f"  Updated {updated} annotations in {files} files\n")
    else:
        print(f"  Warning: {labels_dir} not found, skipping...\n")

print(f"{'='*60}")
print(f"Update complete!")
print(f"Total annotations updated: {total_updated}")
print(f"Total files processed: {total_files}")
print(f"{'='*60}")

# Update the data.yaml in archive (1) to match the new classes
archive_yaml_path = os.path.join(archive_path, 'data.yaml')
with open(archive_yaml_path, 'r') as f:
    archive_yaml = yaml.safe_load(f)

# Update with new class configuration
archive_yaml['nc'] = 36  # Total number of classes in the combined dataset
archive_yaml['names'] = dataset_info['names']

# Save updated data.yaml
with open(archive_yaml_path, 'w') as f:
    yaml.dump(archive_yaml, f, default_flow_style=False, sort_keys=False)

print(f"\nUpdated {archive_yaml_path}")
print(f"New configuration:")
print(f"  nc: {archive_yaml['nc']}")
print(f"  Classes 34-36: {archive_yaml['names'][34:]}")


Original dataset classes:
  0: coffee_nescafe
  1: coffee_kopiko
  2: Lucky-Me-Pancit-Canton
  3: Coke-in-can
  4: Alaska-Milk
  5: Century-Tuna
  6: VCut-Spicy-Barbeque
  7: Selecta-Cornetto
  8: Nestle-Yogurt
  9: Femme-Bathroom-Tissue
  10: Maya-Champorado
  11: JnJ-Potato-Chips
  12: Nivea-Deodorant
  13: UFC-Canned-Mushroom
  14: Libbys-Vienna-Sausage-can
  15: Stik-O
  16: NissinCupNoodles
  17: Dewberry-Strawberry
  18: Smart-C
  19: Pineapple-juice-can
  20: Nestle-Chuckie
  21: Delight-Probiotic-Drink
  22: Summit-Drinking-Water
  23: almond_milk
  24: Piknik
  25: Rambutan
  26: HS-Shampoo
  27: irish-spring-soap
  28: c2_na_green
  29: colgate_toothpaste
  30: 555-sardines
  31: meadows-truffle
  32: double-black
  33: NongshimCupNoodles
  34: Close
  35: Open

Old class mapping (archive dataset):
  0: Rock
  1: Paper
  2: Scissors

New class mapping:
  Rock -> 34
  Paper -> 35
  Scissors -> 36

Starting label update process...

Processing train split...
  Updated 0 annotati

KeyboardInterrupt: 

In [ ]:
from collections import defaultdict

def analyze_class_distribution(archive_path):
    """
    Analyze the distribution of classes across all splits
    
    Args:
        archive_path: Path to the archive directory
    """
    splits = ['train', 'test', 'valid']
    
    print("Class Distribution Analysis:")
    print("=" * 60)
    
    for split in splits:
        labels_dir = os.path.join(archive_path, split, 'labels')
        if not os.path.exists(labels_dir):
            continue
        
        class_counts = defaultdict(int)
        total_annotations = 0
        
        # Count annotations per class
        for label_file in Path(labels_dir).glob("*.txt"):
            with open(label_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        class_counts[class_id] += 1
                        total_annotations += 1
        
        print(f"\n{split.upper()}:")
        print(f"  Total annotations: {total_annotations}")
        print(f"  Class breakdown:")
        
        # Map class IDs to names
        class_names = {34: 'Rock', 35: 'Paper', 36: 'Scissors'}
        
        for class_id in sorted(class_counts.keys()):
            count = class_counts[class_id]
            percentage = (count / total_annotations * 100) if total_annotations > 0 else 0
            class_name = class_names.get(class_id, f'Unknown-{class_id}')
            print(f"    Class {class_id} ({class_name}): {count} ({percentage:.1f}%)")
    
    print("\n" + "=" * 60)

# Run analysis
analyze_class_distribution(archive_path)


## Class Distribution Analysis

Check the distribution of classes after the update:


In [ ]:
import random

def verify_updates(archive_path, num_samples=3):
    """
    Verify that labels were updated correctly by checking sample files
    
    Args:
        archive_path: Path to the archive directory
        num_samples: Number of random samples to check per class
    """
    splits = ['train', 'test', 'valid']
    class_prefixes = ['rock_back', 'paper_back', 'scissors_back']
    expected_classes = [34, 35, 36]  # Expected new class indices
    
    print("Verification Results:")
    print("=" * 60)
    
    for split in splits:
        labels_dir = os.path.join(archive_path, split, 'labels')
        if not os.path.exists(labels_dir):
            print(f"\n{split.upper()} - Directory not found")
            continue
            
        print(f"\n{split.upper()}:")
        
        for prefix, expected_class in zip(class_prefixes, expected_classes):
            # Find files with this prefix
            matching_files = list(Path(labels_dir).glob(f"{prefix}*.txt"))
            
            if not matching_files:
                print(f"  {prefix}: No files found")
                continue
            
            # Sample random files
            sample_files = random.sample(matching_files, min(num_samples, len(matching_files)))
            
            print(f"\n  {prefix} (expected class: {expected_class}):")
            for sample_file in sample_files:
                with open(sample_file, 'r') as f:
                    line = f.readline().strip()
                    if line:
                        class_id = int(line.split()[0])
                        status = "✓" if class_id == expected_class else "✗"
                        print(f"    {status} {sample_file.name}: class {class_id}")
    
    print("\n" + "=" * 60)

# Run verification
verify_updates(archive_path)


## Verification

Verify the updates by checking a few sample label files:


## Update Labels

Run the cell below to update all label files in the dataset:


In [3]:
import shutil
from datetime import datetime

def backup_labels(archive_path, backup_suffix=None):
    """
    Create backup of all label directories
    
    Args:
        archive_path: Path to the archive directory
        backup_suffix: Optional suffix for backup folder (default: timestamp)
    """
    if backup_suffix is None:
        backup_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    backup_root = os.path.join(os.path.dirname(archive_path), f"archive_backup_{backup_suffix}")
    
    splits = ['train', 'test', 'valid']
    for split in splits:
        labels_dir = os.path.join(archive_path, split, 'labels')
        if os.path.exists(labels_dir):
            backup_dir = os.path.join(backup_root, split, 'labels')
            os.makedirs(backup_dir, exist_ok=True)
            
            # Copy all label files
            for label_file in Path(labels_dir).glob("*.txt"):
                shutil.copy2(label_file, backup_dir)
            
            print(f"Backed up {split} labels to {backup_dir}")
    
    # Also backup data.yaml
    yaml_path = os.path.join(archive_path, 'data.yaml')
    if os.path.exists(yaml_path):
        shutil.copy2(yaml_path, os.path.join(backup_root, 'data.yaml'))
        print(f"Backed up data.yaml")
    
    print(f"\nBackup completed at: {backup_root}")
    return backup_root

# Uncomment the line below to create a backup before updating
# backup_path = backup_labels(archive_path)


## Backup Function

Before updating, you can create backups of the original label files:


# Rock Paper Scissors Dataset Label Updater

This notebook updates the class labels in the rock-paper-scissors dataset to match the class indices in the main dataset's `data.yaml` file.

## Class Mapping

The script will update:
- **Rock**: class 0 → class 34
- **Paper**: class 1 → class 35
- **Scissors**: class 2 → class 36 (new class added to dataset)

This allows the rock-paper-scissors dataset to be integrated with the main YOLO dataset that contains 36 classes total.
